# 05-04 LangGraph 多 Agent 协作

**单 Agent 的局限**：一个 Agent 难以同时擅长数据分析、文案创作、合规审核。
多 Agent = 专业分工 + 协作 + 互相检查。

**本节目标**：
- Supervisor 模式（调度员 + 专家 Agent）
- Agent 间通过 State 通信
- 并行 Agent 执行

---

In [ ]:
import os, sys, json
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")
from typing import TypedDict, Literal
from utils.llm_client import call_llm

try:
    from langgraph.graph import StateGraph, END
    HAS_LG = True
except ImportError:
    HAS_LG = False
    print("LangGraph 未安装，展示概念代码")

## 1. 多 Agent 状态定义

In [ ]:
class MultiAgentState(TypedDict):
    """多 Agent 广告优化的共享状态"""
    # 输入
    task: str
    ad_id: str
    
    # 各 Agent 的输出
    data_analysis: str       # 数据分析 Agent 输出
    creative_draft: str      # 创意 Agent 输出
    compliance_check: str    # 合规 Agent 输出
    
    # Supervisor 控制
    next_agent: str          # 下一个执行的 Agent
    final_report: str        # 最终报告

# 专家 Agent 函数
def data_analyst_agent(state: MultiAgentState) -> dict:
    """数据分析 Agent：分析广告效果数据"""
    ad_id = state.get("ad_id", "ad_1")
    try:
        analysis = call_llm(
            f"分析广告 {ad_id} 的效果数据（CTR=3%, CVR=8%, 日消耗750元），给出数据洞察，3句话。",
            system="你是数据分析师",
            max_tokens=200
        )
    except Exception:
        analysis = f"广告{ad_id}：CTR=3%（高于行业均值2.1%），CVR=8%（良好），日消耗750元在预算内。建议在高CTR基础上进一步优化落地页提升CVR。"
    print(f"  [数据分析Agent] {analysis[:60]}...")
    return {"data_analysis": analysis}

def creative_agent(state: MultiAgentState) -> dict:
    """创意 Agent：基于数据分析结果生成优化后的广告文案"""
    analysis = state.get("data_analysis", "")
    try:
        creative = call_llm(
            f"基于以下分析结果，生成优化后的广告标题和正文：\n{analysis}",
            system="你是B站广告文案创作师，标题15字内，正文50字内",
            max_tokens=200
        )
    except Exception:
        creative = "标题：沉浸式游戏体验 限时畅玩\n正文：精选游戏皮肤限时折扣，高清画质极致体验，和百万玩家一起畅玩"
    print(f"  [创意Agent] {creative[:60]}...")
    return {"creative_draft": creative}

def compliance_agent(state: MultiAgentState) -> dict:
    """合规 Agent：检查广告内容合规性"""
    creative = state.get("creative_draft", "")
    issues = []
    for word in ["最", "第一", "绝对", "100%"]:
        if word in creative:
            issues.append(f"含极限词'{word}'")
    result = {"compliant": len(issues) == 0, "issues": issues}
    check = json.dumps(result, ensure_ascii=False)
    print(f"  [合规Agent] {check}")
    return {"compliance_check": check}

def supervisor_agent(state: MultiAgentState) -> dict:
    """Supervisor：决定下一步调用哪个 Agent"""
    if not state.get("data_analysis"):
        print("  [Supervisor] → 先做数据分析")
        return {"next_agent": "data_analyst"}
    elif not state.get("creative_draft"):
        print("  [Supervisor] → 生成创意")
        return {"next_agent": "creative"}
    elif not state.get("compliance_check"):
        print("  [Supervisor] → 合规检查")
        return {"next_agent": "compliance"}
    else:
        report = f"分析: {state['data_analysis'][:50]}... \n创意: {state['creative_draft'][:50]}... \n合规: {state['compliance_check']}"
        print("  [Supervisor] → 任务完成")
        return {"next_agent": "FINISH", "final_report": report}

print("Agent 函数定义完成")

## 2. 构建 Supervisor 模式图

In [ ]:
def route_from_supervisor(state: MultiAgentState) -> str:
    return state.get("next_agent", "FINISH")

if HAS_LG:
    graph = StateGraph(MultiAgentState)
    
    # 添加节点
    graph.add_node("supervisor", supervisor_agent)
    graph.add_node("data_analyst", data_analyst_agent)
    graph.add_node("creative", creative_agent)
    graph.add_node("compliance", compliance_agent)
    
    # 入口 → Supervisor
    graph.set_entry_point("supervisor")
    
    # Supervisor → 条件路由
    graph.add_conditional_edges(
        "supervisor",
        route_from_supervisor,
        {
            "data_analyst": "data_analyst",
            "creative":     "creative",
            "compliance":   "compliance",
            "FINISH":       END,
        }
    )
    
    # 各 Agent → 回到 Supervisor
    graph.add_edge("data_analyst", "supervisor")
    graph.add_edge("creative", "supervisor")
    graph.add_edge("compliance", "supervisor")
    
    app = graph.compile()
    print("多 Agent 图编译成功")
    print("流程: supervisor → agent → supervisor → agent → ... → FINISH")
else:
    print("Supervisor 模式架构:")
    print("  supervisor → data_analyst → supervisor → creative → supervisor → compliance → supervisor → END")

In [ ]:
# 运行多 Agent 工作流
initial = {
    "task": "优化游戏广告投放效果",
    "ad_id": "ad_1",
    "data_analysis": "",
    "creative_draft": "",
    "compliance_check": "",
    "next_agent": "",
    "final_report": "",
}

print("=== 运行多 Agent 广告优化 ===")
if HAS_LG:
    result = app.invoke(initial)
else:
    # 手动模拟
    state = dict(initial)
    for _ in range(5):
        state.update(supervisor_agent(state))
        if state["next_agent"] == "FINISH":
            break
        elif state["next_agent"] == "data_analyst":
            state.update(data_analyst_agent(state))
        elif state["next_agent"] == "creative":
            state.update(creative_agent(state))
        elif state["next_agent"] == "compliance":
            state.update(compliance_agent(state))
    result = state

print(f"\n=== 最终报告 ===")
print(result.get("final_report", "未生成"))

## 多 Agent 架构模式

| 模式 | 说明 | 适用场景 |
|------|------|----------|
| **Supervisor** | 一个调度 Agent 管理多个专家 | 明确分工的任务（如本例） |
| **Hierarchical** | 多层 Supervisor 嵌套 | 大型复杂系统 |
| **Peer-to-peer** | Agent 之间直接对话 | 辩论、审核、协作写作 |
| **Swarm** | Agent 间动态 Handoff | OpenAI Swarm 风格 |

## 面试速记

| 问题 | 要点 |
|------|------|
| 什么时候用多 Agent | 任务需要多种专业能力；需要互相检查；需要并行处理 |
| Agent 间通信方式 | 通过共享 State（LangGraph）或 Message 传递（AutoGen） |
| 如何防止 Agent 循环 | max_iterations 限制 + Supervisor 明确的终止条件 |

**下一节**: `05_langgraph_persistence.ipynb`